# DBT Wrapper - Gold

Runs DBT for the Gold layer with automatic date catch-up and checkpoint persistence.
Shared utilities are loaded from `dbt_wrapper_utils` via `%run`.
Gold-specific pipeline functions are defined in this notebook.

## Execution flow

1. **`setup_gold_execution()`** - declares widgets, validates `checkpoint_file_path`, builds `execution_arguments`.
2. **`resolve_exec_date(extra_flags)`** - extracts `exec_date` from `extra_flags.vars` or defaults to yesterday.
3. **`run_gold_catchup(exec_date, checkpoint_base, dbt_test)`** - for each model_path, runs all missing dates
   from `last_checkpoint + 1` to `exec_date`; updates that model checkpoint after each successful date.
   Optionally runs `dbt test` at the end when `dbt_test=True`.
4. **`run_gold(dbt_test, checkpoint_base, exec_date, table_init)`** - when `table_init=True`, installs deps
   and creates the Gold table from the model pre_hook before running catch-up.

## Parameters

| Parameter | Description |
|---|---|
| `dbt_project` | Path to the DBT project |
| `model_paths` | JSON array of model paths to run in order. Default: `["silver","gold"]` |
| `target` | dbt profile target |
| `properties_file` | Monitoring Exporter properties file path |
| `job_name` / `job_id` / `run_id` | Monitoring log identifiers |
| `extra_flags` | JSON object of extra dbt flags keyed by model_path. Default: `{}` |
| `tags` | JSON object of monitoring tags. Default: `{}` |
| `dbt_test` | Set to `"true"` to run `dbt test` after the model run |
| `checkpoint_file_path` | Base path for checkpoints; one file per model at `<path>/<job_name>/<model_path>/checkpoint.txt` |
| `table_init` | Set to `"true"` on the first run to create the Gold table from the model pre_hook. Default: `"false"` |

### Install DBT

Installs the required DBT libraries and re-starts python to allow for their use.

In [0]:
%pip install dbt-core==1.11.6 dbt-databricks==1.11.6 databricks-sql-connector==4.1.3

In [0]:
dbutils.library.restartPython()

In [0]:
%run ./dbt_wrapper_utils-1.1.1

### Gold Pipeline

Gold-specific functions. Run after `%run ./dbt_wrapper_utils-1.0.0` so they have
access to all shared utilities (`build_dbt_commands`, `run_dbt_stage`,
`get_last_successful_date`, `update_checkpoint_file`, `run_table_init`, etc.).

| Function | Responsibility |
|---|---|
| `setup_gold_execution()` | Declares gold widgets, validates checkpoint_file_path, builds execution_arguments |
| `resolve_exec_date(extra_flags)` | Extracts exec_date from extra_flags.vars or defaults to yesterday |
| `set_exec_date_context(date_to_run, model_checkpoint_file)` | Injects exec_date into extra_info and extra_flags before each run |
| `run_gold_catchup(exec_date, checkpoint_base, dbt_test)` | Per-model date catch-up loop + optional dbt test |
| `run_gold(dbt_test, checkpoint_base, exec_date, table_init)` | Installs deps and creates Gold table when table_init=True, then runs catch-up |

In [0]:
def setup_gold_execution():
    """Declare gold-specific widgets, validate required params, and build execution_arguments.

    Widgets declared here:
        checkpoint_file_path - base path for checkpoint files. One checkpoint file per
                               model is created at
                               <checkpoint_file_path>/<job_name>/<model_path>/checkpoint.txt
        dbt_test             - set to 'true' to run dbt test after the model run.
        table_init           - set to 'true' on the first run to create the Gold table
                               from the model pre_hook.

    Returns:
        (dbt_test: bool, checkpoint_base: str, table_init: bool, execution_args: HashMap)
    """
    dbutils.widgets.text("checkpoint_file_path", "")
    dbutils.widgets.text("dbt_test", "false")
    dbutils.widgets.text("table_init", "false")
    dbt_test = dbutils.widgets.get("dbt_test").lower() == "true"
    table_init = dbutils.widgets.get("table_init").lower() == "true"
    checkpoint_base = dbutils.widgets.get("checkpoint_file_path").strip()
    if not checkpoint_base:
        raise ValueError(
            "checkpoint_file_path parameter is required for Gold Date Catchup."
        )
    checkpoint_base = checkpoint_base.rstrip("/") + "/" + process
    print(f"[INFO] Checkpoint base: {checkpoint_base}")
    execution_args = build_base_execution_arguments(
        model_paths, target, extra_flags
    )
    execution_args.put("dbt_test", dbt_test)
    execution_args.put("checkpoint_base", checkpoint_base)
    execution_args.put("table_init", table_init)
    return dbt_test, checkpoint_base, table_init, execution_args


def resolve_exec_date(extra_flags):
    """Extract the execution date from extra_flags[*]["vars"]["exec_date"].

    If no exec_date is found in any model flags, defaults to yesterday.

    Returns:
        exec_date: date
    """
    exec_date_str = None
    for model_name, flags in extra_flags.items():
        candidate = flags.get("vars", {}).get("exec_date")
        if candidate:
            exec_date_str = candidate
            break
    if not exec_date_str:
        exec_date_str = (
            (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d")
        )
        print(f"exec_date not provided, using yesterday: {exec_date_str}")
    exec_date = datetime.strptime(exec_date_str, "%Y-%m-%d").date()
    print(f"Requested exec_date: {exec_date}")
    return exec_date


def set_exec_date_context(date_to_run, model_checkpoint_file):
    """Inject date_to_run into extra_info and all model extra_flags before model execution."""
    extra_info.put("exec_date", date_to_run.strftime("%Y-%m-%d"))
    extra_info.put("checkpoint_file_path", model_checkpoint_file)
    for model_path in model_paths:
        add_date_flag(model_path, extra_flags, date_to_run.strftime("%Y-%m-%d"))


def run_gold_catchup(exec_date, checkpoint_base, dbt_test=False):
    """Run dbt for all missing dates from each model checkpoint up to exec_date.

    For each model_path: reads its own checkpoint, determines the date range,
    runs each missing date via run_dbt_stage, and updates the checkpoint after
    each successful date. Models with different last checkpointed dates catch up
    independently.
    If exec_date <= checkpoint, the model is already up to date and is skipped.
    If dbt_test is True, runs dbt test after all models finish. A test failure
    emits a warning but does NOT fail the job.

    Raises:
        Exception: on any model failure; the failing model checkpoint is NOT
        updated so the next run retries from there.
    """
    for model_path in model_paths:
        model_checkpoint = checkpoint_base + "/" + model_path + "/checkpoint.txt"
        print(f"[INFO] Checkpoint for '{model_path}': {model_checkpoint}")

        last_successful_date = get_last_successful_date(model_checkpoint)
        if last_successful_date is not None and exec_date <= last_successful_date:
            print(
                f"[INFO] Gold '{model_path}' is up to date "
                f"(checkpoint: {last_successful_date}, exec_date: {exec_date}). Skipping."
            )
            continue
        elif last_successful_date is not None and exec_date > last_successful_date:
            dates_to_run = list(
                generate_date_range(
                    last_successful_date + timedelta(days=1), exec_date
                )
            )
            print(
                f"Catch-up '{model_path}': running {len(dates_to_run)} date(s)"
                f" from {dates_to_run[0]} to {dates_to_run[-1]}"
            )
        else:
            dates_to_run = [exec_date]
            print(f"Running single date for '{model_path}': {exec_date}")

        for date_to_run in dates_to_run:
            print(f"\n--- Running '{model_path}' for date: {date_to_run} ---")
            set_exec_date_context(date_to_run, model_checkpoint)
            commands = build_dbt_commands(
                target, "run", model_path, extra_flags.get(model_path, {})
            )
            run_dbt_stage(
                commands=commands,
                stage=model_path,
                start_msg=f"Running models from '{model_path}' for date {date_to_run}",
                success_msg=f"Ran models from '{model_path}' for date {date_to_run} successfully",
                fail_msg=f"Failed to run models from '{model_path}' for date {date_to_run}"
            )
            update_checkpoint_file(model_checkpoint, date_to_run)

    run_dbt_test(dbt_test)


def run_gold(dbt_test, checkpoint_base, exec_date, table_init=False):
    """Install dbt deps (if packages.yml exists) and run the Gold pipeline.

    table_init=True : creates Gold table from the dbt model pre_hook, then runs catch-up.
                      If a source table is missing, catch-up is skipped and the job
                      succeeds — run again with table_init=False once sources are ready.
    table_init=False: runs catch-up directly.
    """
    install_dbt_deps()
    if table_init:
        run_table_init()
        try:
            run_gold_catchup(exec_date, checkpoint_base, dbt_test)
        except Exception as e:
            if "[TABLE_OR_VIEW_NOT_FOUND]" in str(e):
                print("[INFO] Source table not yet available. Gold table created, catch-up skipped.")
                return
            raise
    else:
        run_gold_catchup(exec_date, checkpoint_base, dbt_test)


In [0]:
dbt_test, checkpoint_base, table_init, execution_arguments = setup_gold_execution()
exec_date = resolve_exec_date(extra_flags)
run_gold(dbt_test, checkpoint_base, exec_date, table_init)
